In [ ]:
!pip install transformers datasets torch scikit-learn pandas openpyxl matplotlib -q

In [14]:
import random
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [15]:
random.seed(42)

form_templates = {
    "filling_form": [
        "Patient has cavity on tooth {tooth} and needs composite filling.",
        "Patient reports sensitivity to cold drinks and filling is needed on tooth {tooth}.",
        "Dental exam showed tooth decay on tooth {tooth}, restorative filling recommended.",
        "Patient says food gets stuck in tooth {tooth}, possible cavity requiring filling.",
        "Tooth {tooth} hurts when eating sweets and may need filling."
    ],
    "extraction_form": [
        "Tooth {tooth} extraction is needed due to severe decay.",
        "Patient has swelling and pain around tooth {tooth}; extraction may be required.",
        "Tooth {tooth} cannot be restored and should be removed.",
        "Broken tooth {tooth} has poor prognosis and extraction is planned.",
        "Patient reports loose painful tooth {tooth}, removal may be needed."
    ],
    "root_canal_form": [
        "Patient requires root canal treatment on tooth {tooth} because of infection.",
        "Severe nerve pain in tooth {tooth}; root canal treatment recommended.",
        "Abscess found near tooth {tooth}; endodontic treatment is needed.",
        "Deep decay reached nerve of tooth {tooth}; root canal therapy planned.",
        "Patient cannot sleep because tooth {tooth} has constant pulsing pain."
    ],
    "cleaning_form": [
        "Patient has plaque buildup and needs dental cleaning.",
        "Bleeding gums observed, cleaning and oral hygiene instructions recommended.",
        "Tartar accumulation found during exam; dental cleaning is required.",
        "Patient is due for regular hygiene cleaning appointment.",
        "Patient reports bad breath and plaque buildup, cleaning recommended."
    ],
    "consultation_form": [
        "Patient reports jaw pain and needs dental consultation.",
        "Patient has general tooth discomfort and requests dental evaluation.",
        "Consultation needed for ongoing mouth pain and sensitivity.",
        "Patient is unsure which tooth hurts and needs examination.",
        "Patient came for general dental assessment and treatment planning."
    ],
    "crown_bridge_form": [
        "Patient needs crown placement on damaged tooth {tooth}.",
        "Old crown on tooth {tooth} is loose and needs replacement.",
        "Bridge replacement required due to fractured dental restoration.",
        "Patient broke a crown while chewing and needs crown evaluation.",
        "Large restoration on tooth {tooth} failed; crown treatment recommended."
    ],
    "periodontal_form": [
        "Patient has gum inflammation and periodontal assessment is required.",
        "Deep pockets and bleeding gums suggest periodontal disease.",
        "Patient reports loose teeth and gum recession, periodontal treatment needed.",
        "Scaling and root planing recommended due to gum disease.",
        "Periodontal charting required because of bleeding and bone loss."
    ],
    "orthodontic_form": [
        "Patient needs orthodontic consultation for crowded teeth.",
        "Braces consultation requested due to misaligned teeth.",
        "Patient has bite issue and may need orthodontic treatment.",
        "Clear aligner assessment requested for teeth alignment.",
        "Patient says teeth are shifting and wants orthodontic advice."
    ],
    "emergency_form": [
        "Patient has severe dental pain and needs emergency appointment.",
        "Patient broke tooth {tooth} yesterday and reports sharp pain.",
        "Facial swelling and severe tooth pain require urgent dental care.",
        "Patient has dental trauma after accident and needs emergency evaluation.",
        "Patient woke up with swollen face and severe mouth pain."
    ]
}

rows = []

for form_type, templates in form_templates.items():
    for _ in range(700):   # 700 per class × 9 classes = 6300
        tooth = random.randint(1, 32)
        text = random.choice(templates).format(tooth=tooth)
        rows.append({"text": text, "label": form_type})

df = pd.DataFrame(rows)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv("iva_role4_improved_dataset.csv", index=False)

print(df.shape)
df["label"].value_counts()

(6300, 2)


,count
label,
root_canal_form,700
filling_form,700
periodontal_form,700
crown_bridge_form,700
extraction_form,700
cleaning_form,700
orthodontic_form,700
consultation_form,700
emergency_form,700


In [16]:
labels = sorted(df["label"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df["label"].map(label2id)

print(label2id)
df.head()

{'cleaning_form': 0, 'consultation_form': 1, 'crown_bridge_form': 2, 'emergency_form': 3, 'extraction_form': 4, 'filling_form': 5, 'orthodontic_form': 6, 'periodontal_form': 7, 'root_canal_form': 8}


,text,label,label_id
0,Abscess found near tooth 19; endodontic treatm...,root_canal_form,8
1,"Patient says food gets stuck in tooth 14, poss...",filling_form,5
2,Patient cannot sleep because tooth 7 has const...,root_canal_form,8
3,Patient has gum inflammation and periodontal a...,periodontal_form,7
4,Tooth 8 hurts when eating sweets and may need ...,filling_form,5


In [17]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_id"],
    random_state=42
)

print(len(train_df), len(val_df), len(test_df))

5040 630 630


In [18]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 5040
    })
    validation: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 630
    })
    test: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 630
    })
})

In [19]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 5040
    })
    validation: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 630
    })
    test: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 630
    })
})

In [20]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [21]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.rename_column("label_id", "labels")

tokenized_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/5040 [00:00<?, ? examples/s]

Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/630 [00:00<?, ? examples/s]

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "f1": f1_score(labels_true, predictions, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./iva_role4_improved_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.265729,0.003753,1.000000,1.000000
2,0.003338,0.001385,1.000000,1.000000
3,0.001807,0.001032,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1890, training_loss=0.09029137476411446, metrics={'train_runtime': 10065.668, 'train_samples_per_second': 1.502, 'train_steps_per_second': 0.188, 'total_flos': 500789275176960.0, 'train_loss': 0.09029137476411446, 'epoch': 3.0})

In [24]:
results = trainer.evaluate(tokenized_dataset["test"])
results

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.00103199842851609,
 'eval_accuracy': 1.0,
 'eval_f1': 1.0,
 'eval_runtime': 130.471,
 'eval_samples_per_second': 4.829,
 'eval_steps_per_second': 0.605,
 'epoch': 3.0}

In [25]:
def predict_form(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)
    predicted_id = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_id].item()

    return {
        "form_type": id2label[predicted_id],
        "confidence": round(confidence, 3)
    }

In [26]:
predict_form("Patient reports severe gum bleeding and deep periodontal pockets.")

{'form_type': 'periodontal_form', 'confidence': 0.999}

In [27]:
predict_form("Patient broke crown on upper molar and needs bridge replacement.")

{'form_type': 'crown_bridge_form', 'confidence': 0.999}

In [28]:
predict_form("Patient arrived with severe swelling and dental emergency pain.")

{'form_type': 'emergency_form', 'confidence': 0.999}

In [29]:
def autofill_form(text, diagnosis=None, procedure=None, medication=None, tooth_number=None):
    prediction = predict_form(text)

    return {
        "form_type": prediction["form_type"],
        "confidence": prediction["confidence"],
        "diagnosis": diagnosis,
        "procedure": procedure,
        "medication": medication,
        "tooth_number": tooth_number,
        "original_text": text,
        "status": "ready_for_review"
    }

In [30]:
autofill_form(
    text="Patient broke crown on tooth 12 and needs crown replacement.",
    diagnosis="broken crown",
    procedure="crown replacement",
    medication=None,
    tooth_number="12"
)

{'form_type': 'crown_bridge_form',
 'confidence': 0.999,
 'diagnosis': 'broken crown',
 'procedure': 'crown replacement',
 'medication': None,
 'tooth_number': '12',
 'original_text': 'Patient broke crown on tooth 12 and needs crown replacement.',
 'status': 'ready_for_review'}